# LeNAHS training

Trains the correction model on a Colab GPU. Datasets and runs are kept in Google
Drive, so a dataset is generated once and an interrupted run resumes.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!git clone -q https://github.com/KadirKess/LeNAHS.git /content/LeNAHS || git -C /content/LeNAHS pull -q
%cd /content/LeNAHS
!pip install -q --no-deps -e .
!pip install -q torch-geometric higra
!git describe --always --dirty

In [ ]:
DRIVE = "/content/drive/MyDrive/LeNAHS"
DATASET = "salt_and_pepper_256"
RUN = "main"
CLEAN_ONLY = False
MAX_EPOCHS = 150

In [ ]:
import os

archive = f"{DRIVE}/datasets/{DATASET}.tar"
dataset = f"/content/datasets/{DATASET}"
os.makedirs(f"{DRIVE}/datasets", exist_ok=True)

if os.path.exists(dataset):
    print("Dataset already extracted")
elif os.path.exists(archive):
    !mkdir -p /content/datasets && tar -xf {archive} -C /content/datasets
else:
    !python scripts/generate_preliminary_dataset.py {dataset}
    !tar -cf {archive} -C /content/datasets {DATASET}

In [ ]:
run = f"{DRIVE}/runs/{RUN}"
flags = "--clean-only" if CLEAN_ONLY else ""
if os.path.exists(f"{run}/last.pt"):
    flags += " --resume"

!python scripts/train_preliminary.py {dataset} {run} --max-epochs {MAX_EPOCHS} --workers 4 {flags}

In [ ]:
import json

import pandas as pd

test = json.load(open(f"{run}/test.json"))
history = json.load(open(f"{run}/history.json"))
quantities = [
    "observed_parent_accuracy",
    "corrective_parent_accuracy",
    "action_accuracy",
    "copy_observed_baseline",
    "always_merge_baseline",
]
groups = [group for group in ("all", "clean", "noisy") if f"{group}/images" in test]

print(f"{len(history)} epochs")
pd.DataFrame(
    {
        (group, weighting): [test[f"{group}/{name}/{weighting}"] for name in quantities]
        for group in groups
        for weighting in ("image_mean", "pooled")
    },
    index=quantities,
).round(3)